# 1 · Sources — one wrapper over every input

`digitalearth.sources` turns any pyramids object (or a raw numpy array) into a uniform `Source` of `(z, x, y, crs, metadata)` — the data layer every plot method reads from. pyramids is the only GIS dependency (no xarray/rasterio).

The sample data is a **real elevation DEM of the Lisbon region** (421×547 cells, EPSG:4326).

In [1]:
%matplotlib inline
from pathlib import Path

# Resolve the repo root so the bundled sample DEM is found whether this runs from
# docs/examples/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DEM = str(ROOT / "examples" / "data" / "LisbonElevation.tif")
print("DEM:", DEM)

DEM: C:\gdrive\algorithms\Visualization\Digital-Earth\examples\data\LisbonElevation.tif


## From a pyramids `Dataset` (the DEM raster)

In [2]:
from pyramids.dataset import Dataset
from digitalearth.sources import get_source

ds = Dataset.read_file(DEM)
src = get_source(ds)
print('kind     :', src.metadata('kind'))
print('z shape  :', src.z.values.shape)
print('x / y len:', src.x.values.shape, src.y.values.shape)
print('crs      :', src.crs)

2026-05-28 19:58:00 | INFO | pyramids.base.config | Logging is configured.


kind     : raster
z shape  : (421, 547)
x / y len: (547,) (421,)
crs      : 4326


The data dimension is NaN-masked at the nodata value; `x`/`y` are 1-D cell-centre coordinates:

In [3]:
import numpy as np
z = src.z.values
print('elevation range (m): %.1f .. %.1f' % (np.nanmin(z), np.nanmax(z)))
print('lon range:', round(float(src.x.values.min()), 3), '..', round(float(src.x.values.max()), 3))
print('lat range:', round(float(src.y.values.min()), 3), '..', round(float(src.y.values.max()), 3))

elevation range (m): -6.8 .. 217.9
lon range: -9.238 .. -9.086
lat range: 38.68 .. 38.797


## From a raw numpy array

No CRS; pixel-index axes unless you pass `x`/`y`.

In [4]:
arr = np.arange(12.0).reshape(3, 4)
s = get_source(arr)
print('x:', s.x.values.tolist(), ' crs:', s.crs)

x: [0.0, 1.0, 2.0, 3.0]  crs: None


## From a `DatasetCollection` (one member)

In [5]:
from pyramids.dataset.collection import DatasetCollection

dc = DatasetCollection.from_files([DEM, DEM])
sc = get_source(dc)
print('member / n_members:', sc.metadata('member'), '/', sc.metadata('n_members'))

member / n_members: 0 / 2
